# Stage 1 Go/No-Go Experiment (PMB vs baselines under identity corruption)

Run the cells top to bottom (Runtime → Run all).

1. Install deps
2. Write the 7 Python files into a `stage1/` package (the code imports `stage1.xxx`)
3. Run the experiment on **synthetic, CICIoT2023-shaped data** (validates the pipeline only, not the real Stage 1 answer)
4. Optional: multi-seed run for a less noisy read

In [ ]:
!pip -q install numpy scipy scikit-learn

## 1. Create the package

In [ ]:
import os
os.makedirs('stage1', exist_ok=True)
open('stage1/__init__.py','w').close()

In [ ]:
%%writefile stage1/corruption.py
# -*- coding: utf-8 -*-
"""
Identity corruption simulator for the Stage 1 go/no-go experiment.

Implements the 5 corruption knobs from the roadmap:
  - MAC rotation rate: device identifier changes over time
  - Hub/NAT aggregation size: N devices merged into one observed stream
  - Spoof-duplicate fraction: devices sharing an identifier
  - Hidden/unknown-device fraction: devices with no usable identifier
  - Fingerprinting error rate: identity-resolution stage assigns wrong device

Each knob is applied independently so you can vary them one at a time
(as the roadmap specifies) or combined.
"""
from dataclasses import dataclass
import numpy as np


@dataclass
class CorruptionConfig:
    mac_rotation_rate: float = 0.0       # prob. a device's ID changes at each window
    hub_aggregation_size: int = 1        # how many true devices share one observed ID
    spoof_duplicate_frac: float = 0.0    # fraction of devices that share an ID w/ another
    hidden_device_frac: float = 0.0      # fraction of devices with no usable ID (None)
    fingerprint_error_rate: float = 0.0  # used only by B1's resolver, not here


def apply_corruption(records, config: CorruptionConfig, rng: np.random.Generator):
    """
    records: list of dicts, each with at least:
        {'true_device_id': int, 'window': int, 'features': np.ndarray, 'is_attack': bool}
    Returns a NEW list of dicts with an added 'observed_id' field (str or None),
    'true_device_id' is preserved for evaluation only -- the detector/tracker
    must never be given 'true_device_id' directly.
    """
    n_true_devices = len({r['true_device_id'] for r in records})

    # --- hub aggregation: map groups of true devices to one hub id ---
    device_ids = sorted({r['true_device_id'] for r in records})
    hub_map = {}
    if config.hub_aggregation_size > 1:
        for i, dev in enumerate(device_ids):
            hub_map[dev] = f"hub_{i // config.hub_aggregation_size}"
    else:
        hub_map = {dev: f"dev_{dev}" for dev in device_ids}

    # --- spoof duplicates: some devices forced to share another device's id ---
    spoof_map = {}
    n_spoofed = int(len(device_ids) * config.spoof_duplicate_frac)
    spoof_targets = rng.choice(device_ids, size=n_spoofed, replace=False) if n_spoofed else []
    for dev in spoof_targets:
        victim = rng.choice([d for d in device_ids if d != dev])
        spoof_map[dev] = hub_map[victim]

    # --- hidden devices: never get a usable id ---
    n_hidden = int(len(device_ids) * config.hidden_device_frac)
    hidden_devices = set(rng.choice(device_ids, size=n_hidden, replace=False)) if n_hidden else set()

    # --- per-device rotating alias state for MAC rotation ---
    rotation_state = {dev: hub_map[dev] for dev in device_ids}

    out = []
    for r in records:
        dev = r['true_device_id']
        rec = dict(r)

        if dev in hidden_devices:
            rec['observed_id'] = None
            out.append(rec)
            continue

        # MAC rotation: with given probability, this device gets a fresh alias
        if config.mac_rotation_rate > 0 and rng.random() < config.mac_rotation_rate:
            rotation_state[dev] = f"{hub_map[dev]}_rot{rng.integers(0, 1_000_000)}"

        observed = rotation_state[dev]

        # spoof duplication overrides with the shared/victim id
        if dev in spoof_map:
            observed = spoof_map[dev]

        rec['observed_id'] = observed
        out.append(rec)

    return out

In [ ]:
%%writefile stage1/detector.py
# -*- coding: utf-8 -*-
"""
A lightweight, swappable anomaly-score detector.

Stage 1 is testing whether identity-aware tracking (P1) recovers performance
lost to identity corruption -- NOT testing which detector is best. So this
file deliberately uses a simple, fast, well-understood scorer (IsolationForest
on per-window features) as a stand-in for the real Paper-1 GRU/POMDP detector.

To use your real detector later: replace `score_windows()` with a call into
your trained GRU model, keeping the same input/output shape (features in,
one anomaly score in [0, 1] out per window). Nothing else in Stage 1 needs
to change.
"""
import numpy as np
from sklearn.ensemble import IsolationForest


def fit_detector(feature_matrix: np.ndarray, random_state=0):
    """Fits on a feature matrix assumed to be mostly-normal traffic."""
    clf = IsolationForest(
        n_estimators=150,
        contamination="auto",
        random_state=random_state,
    )
    clf.fit(feature_matrix)
    return clf


def score_windows(clf, feature_matrix: np.ndarray) -> np.ndarray:
    """
    Returns anomaly scores in [0, 1], higher = more anomalous.
    IsolationForest's decision_function is higher-for-normal, so we flip and
    min-max normalize for a consistent 0..1 anomaly convention.
    """
    raw = clf.decision_function(feature_matrix)  # higher = more normal
    inv = -raw
    lo, hi = inv.min(), inv.max()
    if hi - lo < 1e-12:
        return np.zeros_like(inv)
    return (inv - lo) / (hi - lo)


def fingerprint_resolve(records, error_rate: float, rng: np.random.Generator):
    """
    Simulates B1's identity-resolution stage: takes corrupted observed_ids
    and produces a "resolved" id per record, imperfectly. A correct resolver
    would recover the rotated/hub-merged identity; `error_rate` is the
    fraction of records where it fails and assigns a wrong/fresh label
    instead, modeling real fingerprinting error.
    """
    out = []
    for r in records:
        rec = dict(r)
        obs = rec['observed_id']
        if obs is None:
            rec['resolved_id'] = f"unresolved_{rng.integers(0, 1_000_000)}"
        elif rng.random() < error_rate:
            rec['resolved_id'] = f"misresolved_{rng.integers(0, 1_000_000)}"
        else:
            rec['resolved_id'] = obs
        out.append(rec)
    return out

In [ ]:
%%writefile stage1/baselines.py
# -*- coding: utf-8 -*-
"""
B0-B3 baselines from the roadmap's Stage 1 methods table.

B0: oracle (ground-truth) IDs, upper bound.
B1: identity-first (fingerprint/cluster to hard IDs, then per-device).
B2: network-level/aggregate, identity-free.
B3: independent per-device Bernoulli on the (possibly wrong) raw observed IDs
    -- naive, no birth model, no clutter handling. Tests whether the PMB's
    extra machinery is needed at all, or independent-per-id tracking is enough.
"""
from collections import defaultdict


def run_b0_oracle(records_scored):
    """records_scored: list of dicts with true_device_id and anomaly_score.
    Returns {true_device_id: set of windows flagged compromised}."""
    flagged = defaultdict(set)
    for r in records_scored:
        if r['anomaly_score'] >= 0.5:
            flagged[r['true_device_id']].add(r['window'])
    return flagged


def run_b1_identity_first(records_scored, threshold=0.5):
    """Uses resolved_id (from detector.fingerprint_resolve)."""
    flagged = defaultdict(set)
    for r in records_scored:
        if r['anomaly_score'] >= threshold:
            flagged[r['resolved_id']].add(r['window'])
    return flagged


def run_b2_aggregate(records_scored, threshold=0.5):
    """No identity at all -- one global stream. Returns just the set of
    windows where *something* looked compromised (no per-device breakdown
    possible by construction, which is exactly B2's limitation)."""
    flagged_windows = {r['window'] for r in records_scored if r['anomaly_score'] >= threshold}
    return flagged_windows


def run_b3_naive_per_id(records_scored, threshold=0.5):
    """Uses raw observed_id directly, no birth model, no clutter filter --
    every id that ever crosses the threshold is immediately 'compromised',
    every unseen id silently stops existing (no decay/death modeling)."""
    flagged = defaultdict(set)
    for r in records_scored:
        if r['observed_id'] is None:
            continue
        if r['anomaly_score'] >= threshold:
            flagged[r['observed_id']].add(r['window'])
    return flagged

In [ ]:
%%writefile stage1/pmb_filter.py
# -*- coding: utf-8 -*-
"""
Simplified, single-hypothesis Poisson Multi-Bernoulli (PMB) filter -- the
P1 / proposed method from the roadmap.

This is deliberately a SIMPLIFIED PMB, not a full PMBM (no multi-hypothesis
data association). It matches the structural choice noted in Update 5:
an explicit Poisson pool for "not yet confirmed" observations, and a
confirmed Bernoulli track is only spawned once evidence (repeated
observation + anomaly score) justifies it -- rather than pre-populating
every possible id with existence probability from step one (which Update 5
identified as LMB's likely weakness in the earlier unfair comparison).

Honest scope note: this omits full RFS machinery (no OSPA-optimal
multi-hypothesis association, no proper Bayes recursion over state
densities). It's built to test the GO/NO-GO question in the roadmap --
does *tracking identity probabilistically* recover performance lost to
corruption -- not to be the final, paper-ready filter. If Stage 1 passes,
upgrading this to a real PMBM implementation is the next step, per the
roadmap's own "planned upgrade path."
"""
from dataclasses import dataclass, field
import numpy as np


@dataclass
class Track:
    label: str
    r: float                 # existence probability
    compromise_belief: float # P(this device is compromised)
    last_seen: int
    born_at: int
    hits: int = 1


@dataclass
class PMBConfig:
    birth_threshold: float = 0.5     # anomaly score needed to seed a new track from the Poisson pool
    birth_r: float = 0.3             # initial existence prob for a newly spawned track
    confirm_hits: int = 2            # observations needed before a track counts as "confirmed"
    decay_per_window: float = 0.15   # existence prob decays this much per window unseen
    death_threshold: float = 0.05    # track removed once r drops below this
    update_gain: float = 0.4         # how much one observation moves compromise_belief


class PMBFilter:
    def __init__(self, config: PMBConfig):
        self.cfg = config
        self.tracks: dict[str, Track] = {}

    def step(self, window: int, observations: list[tuple[str, float]]):
        """
        observations: list of (observed_id, anomaly_score) seen in this window.
        observed_id may repeat within a window if hub-aggregated; that's fine,
        each is processed independently, matching normal RFS measurement handling.
        """
        seen_this_window = set()

        for obs_id, score in observations:
            seen_this_window.add(obs_id)
            if obs_id in self.tracks:
                t = self.tracks[obs_id]
                t.hits += 1
                t.last_seen = window
                t.r = min(1.0, t.r + (1 - t.r) * 0.5)  # re-observation raises existence confidence
                t.compromise_belief = (
                    (1 - self.cfg.update_gain) * t.compromise_belief
                    + self.cfg.update_gain * score
                )
            else:
                # Poisson pool decision: does this look enough like a real,
                # noteworthy object to spawn a confirmed-candidate Bernoulli track?
                if score >= self.cfg.birth_threshold:
                    self.tracks[obs_id] = Track(
                        label=obs_id,
                        r=self.cfg.birth_r,
                        compromise_belief=score,
                        last_seen=window,
                        born_at=window,
                    )
                # else: treated as clutter, discarded, no track created

        # decay / death for tracks not seen this window
        dead = []
        for label, t in self.tracks.items():
            if label not in seen_this_window:
                t.r -= self.cfg.decay_per_window
                if t.r < self.cfg.death_threshold:
                    dead.append(label)
        for label in dead:
            del self.tracks[label]

    def confirmed_compromised(self, compromise_threshold: float = 0.5):
        """Returns the set of track labels currently believed compromised."""
        return {
            t.label for t in self.tracks.values()
            if t.hits >= self.cfg.confirm_hits and t.compromise_belief >= compromise_threshold
        }

    def cardinality_estimate(self):
        """Expected number of existing objects = sum of existence probabilities."""
        return sum(t.r for t in self.tracks.values())

In [ ]:
%%writefile stage1/metrics.py
# -*- coding: utf-8 -*-
"""
Metrics for the Stage 1 go/no-go experiment.

Implements: cardinality error, per-window detection F1 (mapped back to true
devices via majority vote, since observed/resolved labels aren't the true
device id), and false isolation rate. A full OSPA metric is NOT implemented
here (noted in the roadmap as still needing metric-choice confirmation) --
label_to_true_device() below is the piece a real OSPA implementation would
also need, so it's built to be reused.
"""
from collections import defaultdict, Counter
import numpy as np


def label_to_true_device_map(records):
    """
    Builds {observed_or_resolved_label: majority true_device_id it maps to}.
    This uses ground truth and must ONLY be used for evaluation, never fed
    to a detector/tracker.
    """
    votes = defaultdict(Counter)
    for r in records:
        for key in ('observed_id', 'resolved_id'):
            if key in r and r[key] is not None:
                votes[r[key]][r['true_device_id']] += 1
    return {label: counter.most_common(1)[0][0] for label, counter in votes.items()}


def cardinality_mae(estimated_per_window: dict, true_per_window: dict):
    windows = sorted(set(estimated_per_window) | set(true_per_window))
    errs = [abs(estimated_per_window.get(w, 0) - true_per_window.get(w, 0)) for w in windows]
    return float(np.mean(errs)) if errs else float('nan')


def true_compromised_per_window(records):
    """{window: set(true_device_id) that are actually compromised in that window}."""
    out = defaultdict(set)
    for r in records:
        if r['is_attack']:
            out[r['window']].add(r['true_device_id'])
    return out


def detection_prf1(flagged_by_label: dict, id_map: dict, true_per_window: dict, clean_devices: set):
    """
    flagged_by_label: {label: set(windows flagged)}  -- from a baseline/P1 run
    id_map: label -> true_device_id (majority vote, eval-only)
    true_per_window: {window: set(true compromised device ids)}
    clean_devices: set of true_device_id that are NEVER actually compromised
                   (used for false isolation rate)

    Returns dict with precision, recall, f1, false_isolation_rate.
    """
    tp = fp = fn = 0
    false_isolations = 0
    all_windows = sorted(true_per_window)

    for w in all_windows:
        predicted_devices = {
            id_map.get(label, label)  # fall back to raw label if unmapped
            for label, windows in flagged_by_label.items() if w in windows
        }
        true_devices = true_per_window.get(w, set())

        tp += len(predicted_devices & true_devices)
        fp += len(predicted_devices - true_devices)
        fn += len(true_devices - predicted_devices)
        false_isolations += len(predicted_devices & clean_devices)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    fir = false_isolations / (len(clean_devices) * len(all_windows)) if clean_devices and all_windows else 0.0

    return {
        'precision': precision, 'recall': recall, 'f1': f1,
        'false_isolation_rate': fir,
    }

In [ ]:
%%writefile stage1/data.py
# -*- coding: utf-8 -*-
"""
Data loading for Stage 1.

`generate_synthetic()` builds a CICIoT2023-SHAPED dataset (same record
structure: true_device_id, window, features, is_attack) purely to validate
the pipeline logic end-to-end without needing the real dataset downloaded.
Results from synthetic data tell you the CODE WORKS -- they are NOT a
substitute for the real go/no-go answer, which needs real CICIoT2023 data.

`load_ciciot2023()` is a documented stub: fill in the actual column names
once you have the CSV(s) locally. The rest of the pipeline only needs the
four fields below, so once this function returns the right shape, nothing
else in stage1/ needs to change.
"""
import numpy as np


def generate_synthetic(
    n_devices=40,
    n_windows=200,
    n_features=8,
    attack_devices_frac=0.25,
    attack_start_frac=0.5,
    seed=0,
):
    """
    Builds a toy dataset with the same shape as CICIoT2023 per-window
    features. A subset of devices become "compromised" partway through
    (features shift + is_attack flag), the rest stay normal throughout.
    """
    rng = np.random.default_rng(seed)
    n_attack_devices = int(n_devices * attack_devices_frac)
    attack_devices = set(rng.choice(n_devices, size=n_attack_devices, replace=False).tolist())
    attack_start_window = int(n_windows * attack_start_frac)

    records = []
    for dev in range(n_devices):
        base = rng.normal(0, 1, size=n_features)  # this device's normal traffic "fingerprint"
        for w in range(n_windows):
            is_attack = dev in attack_devices and w >= attack_start_window
            if is_attack:
                feat = base + rng.normal(3.0, 0.8, size=n_features)  # shifted = anomalous
            else:
                feat = base + rng.normal(0, 0.5, size=n_features)
            records.append({
                'true_device_id': dev,
                'window': w,
                'features': feat,
                'is_attack': bool(is_attack),
            })
    return records


def load_ciciot2023(csv_path: str):
    """
    STUB -- fill in once you have the real file(s) locally.

    Expected minimum columns in CICIoT2023's per-window feature files:
      - a device identifier column (dataset provides per-device captures;
        check whether device id is a filename/column or must be inferred)
      - a window/flow index (or derive sequential windows per device)
      - the 47 pre-extracted numeric features
      - the attack label column (map multi-class labels to a boolean
        is_attack for Stage 1; keep the original label separately if you
        want per-attack-family breakdowns later)

    Must return: list of dicts, each
        {'true_device_id': <hashable>, 'window': <int>, 'features': np.ndarray, 'is_attack': bool}
    matching generate_synthetic()'s output shape exactly.
    """
    raise NotImplementedError(
        "Fill in CICIoT2023 column mapping here once the CSV(s) are available locally. "
        "See the docstring above for the required output shape."
    )

In [ ]:
%%writefile stage1/run_stage1.py
# -*- coding: utf-8 -*-
"""
Stage 1 go/no-go experiment runner.

Usage:
    python run_stage1.py                 # dry run on synthetic data (validates the pipeline)
    python run_stage1.py --real PATH     # once load_ciciot2023() in data.py is filled in

This implements the comparison table from the roadmap:
    B0 = oracle ids (upper bound)
    B1 = identity-first (fingerprint/cluster, then per-device)
    B2 = aggregate, identity-free
    B3 = naive independent per-id Bernoulli (no birth model)
    P1 = PMB filter (Poisson pool + confirmed Bernoulli tracks) -- proposed method

Applies the roadmap's own pre-registered decision rule at the end:
    GO         if B1 loses a lot of B0's performance AND P1 recovers >= ~half the gap
               AND P1 clearly beats B3
    DOWNGRADE  if B3 matches P1 (identity ambiguity real, but PMB machinery adds nothing)
    NO-GO      if B1 degrades only gracefully (problem not severe enough)

These thresholds are editable at the top of main() -- the roadmap says to
set them before looking at results, so do that before running on real data.
"""
import argparse
import numpy as np
from collections import defaultdict

from stage1.corruption import CorruptionConfig, apply_corruption
from stage1.detector import fit_detector, score_windows, fingerprint_resolve
from stage1 import baselines
from stage1.pmb_filter import PMBFilter, PMBConfig
from stage1 import metrics
from stage1.data import generate_synthetic


def run_one_condition(clean_records, corruption_cfg: CorruptionConfig, rng, fp_error_rate: float):
    # 1. corrupt identities
    corrupted = apply_corruption(clean_records, corruption_cfg, rng)

    # 2. fit + score with the (swappable) detector, using only normal-looking
    #    early windows to fit, matching realistic "train on mostly-normal" setup
    feat_matrix = np.array([r['features'] for r in corrupted])
    n_fit = max(50, len(corrupted) // 4)
    clf = fit_detector(feat_matrix[:n_fit])
    scores = score_windows(clf, feat_matrix)
    for r, s in zip(corrupted, scores):
        r['anomaly_score'] = float(s)

    # 3. B1 needs a resolved id (imperfect fingerprinting)
    resolved = fingerprint_resolve(corrupted, error_rate=fp_error_rate, rng=rng)

    # 4. run baselines
    b0 = baselines.run_b0_oracle(resolved)
    b1 = baselines.run_b1_identity_first(resolved)
    b3 = baselines.run_b3_naive_per_id(resolved)

    # 5. run P1 (PMB) window by window, in time order
    pmb = PMBFilter(PMBConfig())
    by_window = defaultdict(list)
    for r in resolved:
        if r['observed_id'] is not None:
            by_window[r['window']].append((r['observed_id'], r['anomaly_score']))
    est_cardinality_per_window = {}
    p1_flagged = defaultdict(set)
    for w in sorted(by_window):
        pmb.step(w, by_window[w])
        est_cardinality_per_window[w] = pmb.cardinality_estimate()
        for label in pmb.confirmed_compromised():
            p1_flagged[label].add(w)

    # 6. evaluation setup
    id_map = metrics.label_to_true_device_map(resolved)
    true_pw = metrics.true_compromised_per_window(resolved)
    true_cardinality_pw = {w: len(devs) for w, devs in true_pw.items()}
    all_true_devices = {r['true_device_id'] for r in resolved}
    ever_compromised = {r['true_device_id'] for r in resolved if r['is_attack']}
    clean_devices = all_true_devices - ever_compromised

    def eval_baseline(flagged):
        f1 = metrics.detection_prf1(flagged, id_map, true_pw, clean_devices)
        card_est = {w: len(devs) for w, devs in flagged.items()}
        return f1

    results = {
        'B0': eval_baseline(b0),
        'B1': eval_baseline(b1),
        'B3': eval_baseline(b3),
        'P1': eval_baseline(p1_flagged),
    }
    results['P1']['cardinality_mae'] = metrics.cardinality_mae(est_cardinality_per_window, true_cardinality_pw)

    return results


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--real', type=str, default=None, help='Path to real CICIoT2023 data (requires filling in data.load_ciciot2023 first)')
    parser.add_argument('--seed', type=int, default=0)
    args = parser.parse_args()

    rng = np.random.default_rng(args.seed)

    if args.real:
        from stage1.data import load_ciciot2023
        clean_records = load_ciciot2023(args.real)
        print(f"Loaded {len(clean_records)} real records from {args.real}")
    else:
        clean_records = generate_synthetic(seed=args.seed)
        print(f"DRY RUN on synthetic data ({len(clean_records)} records). "
              f"This validates the pipeline only -- not a real go/no-go answer.")

    corruption_levels = [
        CorruptionConfig(mac_rotation_rate=0.0, hub_aggregation_size=1, spoof_duplicate_frac=0.0, hidden_device_frac=0.0),
        CorruptionConfig(mac_rotation_rate=0.15, hub_aggregation_size=1, spoof_duplicate_frac=0.1, hidden_device_frac=0.05),
        CorruptionConfig(mac_rotation_rate=0.4, hub_aggregation_size=3, spoof_duplicate_frac=0.25, hidden_device_frac=0.15),
    ]
    labels = ['clean (0.0)', 'moderate corruption', 'heavy corruption']

    print(f"\n{'condition':<22}{'method':<6}{'precision':<12}{'recall':<10}{'f1':<8}{'false_iso_rate':<16}")
    all_results = {}
    for label, cfg in zip(labels, corruption_levels):
        res = run_one_condition(clean_records, cfg, rng, fp_error_rate=0.15)
        all_results[label] = res
        for method, m in res.items():
            print(f"{label:<22}{method:<6}{m['precision']:<12.3f}{m['recall']:<10.3f}{m['f1']:<8.3f}{m['false_isolation_rate']:<16.3f}")

    # --- decision rule, thresholds set here BEFORE reading results, per the roadmap ---
    RECOVERY_FRACTION_NEEDED = 0.5   # P1 must recover >= 50% of B0-B1 gap
    B3_MARGIN_NEEDED = 0.05          # P1's f1 must beat B3's f1 by at least this much
    heavy = all_results['heavy corruption']
    b0_f1, b1_f1, b3_f1, p1_f1 = heavy['B0']['f1'], heavy['B1']['f1'], heavy['B3']['f1'], heavy['P1']['f1']
    gap = b0_f1 - b1_f1
    recovered = (p1_f1 - b1_f1) / gap if gap > 1e-9 else float('nan')

    print(f"\n--- Decision rule (heavy corruption condition) ---")
    print(f"B0={b0_f1:.3f}  B1={b1_f1:.3f}  B3={b3_f1:.3f}  P1={p1_f1:.3f}")
    print(f"B0-B1 gap: {gap:.3f} | P1 recovers: {recovered:.1%} of that gap | P1 vs B3 margin: {p1_f1 - b3_f1:.3f}")

    if p1_f1 - b3_f1 < B3_MARGIN_NEEDED:
        verdict = "DOWNGRADE to Level 1 (B3 matches P1 -- PMB machinery not earning its keep)"
    elif gap > 0.05 and recovered >= RECOVERY_FRACTION_NEEDED and (p1_f1 - b3_f1) >= B3_MARGIN_NEEDED:
        verdict = "GO"
    else:
        verdict = "NO-GO (B1 degrades gracefully -- problem not severe enough on this data)"

    print(f"\nVERDICT: {verdict}")
    if not args.real:
        print("\n(Remember: this verdict is on SYNTHETIC data and only proves the pipeline runs "
              "correctly end-to-end. Fill in data.load_ciciot2023() and re-run with --real "
              "before treating any verdict as the actual Stage 1 answer.)")


if __name__ == '__main__':
    main()

## 2. Run the experiment (synthetic dry run)

In [ ]:
!python -m stage1.run_stage1 --seed 0

## 3. Multi-seed run (optional)
A single seed is noisy. This repeats the heavy-corruption condition over several seeds and reports mean ± std of F1 per method,
then applies the same decision rule to the means.

In [ ]:
import numpy as np
from stage1.data import generate_synthetic
from stage1.corruption import CorruptionConfig
from stage1.run_stage1 import run_one_condition

N_SEEDS = 10
heavy_cfg = CorruptionConfig(mac_rotation_rate=0.4, hub_aggregation_size=3,
                             spoof_duplicate_frac=0.25, hidden_device_frac=0.15)
f1 = {m: [] for m in ['B0', 'B1', 'B3', 'P1']}
for s in range(N_SEEDS):
    rng = np.random.default_rng(s)
    recs = generate_synthetic(seed=s)
    res = run_one_condition(recs, heavy_cfg, rng, fp_error_rate=0.15)
    for m in f1:
        f1[m].append(res[m]['f1'])

print(f"Heavy corruption, {N_SEEDS} seeds (F1 mean ± std)")
for m, v in f1.items():
    print(f"  {m}: {np.mean(v):.3f} ± {np.std(v):.3f}")

b0, b1, b3, p1 = (np.mean(f1[m]) for m in ['B0', 'B1', 'B3', 'P1'])
gap = b0 - b1
rec = (p1 - b1) / gap if gap > 1e-9 else float('nan')
print(f"\nB0-B1 gap: {gap:.3f} | P1 recovers {rec:.1%} of gap | P1 - B3 margin: {p1 - b3:.3f}")

## Notes
- Results on synthetic data only show the code runs end to end. The real Stage 1 answer needs `load_ciciot2023()` in `stage1/data.py` filled in.
- Set your decision thresholds (`RECOVERY_FRACTION_NEEDED`, `B3_MARGIN_NEEDED` in `stage1/run_stage1.py`) **before** looking at real-data results.
- P1 here is a simplified single-hypothesis PMB, not full PMBM.